In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import *

In [0]:
transactions_df = spark.read.table("bankaml.silver.transactions")
accounts_df = spark.read.table("bankaml.silver.accounts").filter(col("is_current") == "Y")

In [0]:
IN_TXN_TYPES = ['wire_in', 'deposit', 'transfer_in',]
OUT_TXN_TYPES = ['wire_out', 'withdrawal',  'transfer_out']
OUT_FLOW_THRESHOLD = 0.85

inflows_df = transactions_df.filter(col("txn_type").isin(IN_TXN_TYPES)).select("txn_id", "account_id", "txn_type", "amount_usd", "txn_ts")
outflows_df = transactions_df.filter(col("txn_type").isin(OUT_TXN_TYPES)).select("txn_id", "account_id", "txn_type", "amount_usd", "txn_ts")

paired_df = (
    inflows_df.alias("i").join(
        outflows_df.alias("o"), 
        (
            ((col("i.account_id")) == (col("o.account_id")))
            & (col("o.txn_ts") > col("i.txn_ts"))
            & (col("o.txn_ts") <= (col("i.txn_ts") + expr("INTERVAL 24 HOURS")))
            & (col("o.amount_usd") >= (col("i.amount_usd") * IN_OUT_FLOW_THRESHOLD))
        )
        , "inner"
    )
    .withColumn("time_gap_hours", round(
        ((col("o.txn_ts").cast(LongType())
          - col("i.txn_ts").cast(LongType()))
         /3600)
        , 2)
    )
    .withColumn("flag_type", lit("rapid_in_out"))
    .withColumn("severity", lit("high"))
    .withColumn("flagged_ts", current_timestamp())
    .withColumn("flag_id", sha2(concat_ws("_", "i.txn_id", "o.txn_id"), 256))
)
rapid_in_out_df = paired_df.select(
    col("i.account_id").alias("account_id"),
    col("i.txn_id").alias("inflow_txn_id"),
    col("o.txn_id").alias("outflow_txn_id"),
    col("i.amount_usd").alias("inflow_amount_usd"),
    col("o.amount_usd").alias("outflow_amount_usd"),
    col("time_gap_hours"),
    col("flag_type"),
    col("severity"),
    col("flagged_ts"),
    col("flag_id")
)

rapid_in_out_df = (
    rapid_in_out_df.alias("r").join(
        accounts_df.alias("a"), 
        col("r.account_id") == col("a.account_id"), 
        "inner"
    ).select(
        "r.*", 
        col("a.customer_id").alias("customer_id")
    )
)

In [0]:
%sql
create table if not exists bankaml.gold.rapid_inout_flags
(
    account_id string,
    customer_id string,
    inflow_txn_id string,
    outflow_txn_id string,
    inflow_amount_usd decimal(18,2),
    outflow_amount_usd decimal(18,2),
    time_gap_hours decimal(10,2),
    flag_type string,
    severity string,
    flagged_ts timestamp,
    flag_id string
)

In [0]:
rapid_inout_flags_table = DeltaTable.forName(spark, "bankaml.gold.rapid_inout_flags")
rapid_in_out_df = rapid_in_out_df.select(*rapid_inout_flags_table.toDF().columns)

(
    rapid_inout_flags_table.alias("t").merge(
        rapid_in_out_df.alias("s"),
        "t.flag_id=s.flag_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)